In [0]:
# ── Gold: Bench Intelligence ──────────────────────────────────────────────────
# Reads from silver.bench_periods, silver.employees
# - Current bench headcount and average bench duration
# - Bench cost by practice area and seniority
# - Long term bench flags
# - Unexpected bench flags for Managers and Senior Managers
# ─────────────────────────────────────────────────────────────────────────────

spark.sql("USE CATALOG workspace")

from pyspark.sql.functions import (
    col, round, avg, sum, count, when, lit,
    current_timestamp, max, min, datediff
)

# Read from silver
bench_df     = spark.table("silver.bench_periods")
employees_df = spark.table("silver.employees")

print(f"Silver bench periods: {bench_df.count()}")
print(f"Silver employees:     {employees_df.count()}")
bench_df.printSchema()

In [0]:
# ── Build bench intelligence ───────────────────────────────────────────────────
spark.sql("USE CATALOG workspace")

bench_df     = spark.table("silver.bench_periods")
employees_df = spark.table("silver.employees")

joined_df = (
    bench_df
    .join(employees_df.select("employee_id", "practice_area", "name"),
          on="employee_id", how="left")
)

# Employee level bench summary
employee_bench_df = (
    joined_df
    .groupBy("employee_id", "name", "seniority", "practice_area")
    .agg(
        count("bench_id").alias("total_bench_periods"),
        round(sum("days_on_bench_clean"), 0).alias("total_days_on_bench"),
        round(avg("days_on_bench_clean"), 0).alias("avg_days_per_bench"),
        round(sum("estimated_bench_cost"), 2).alias("total_estimated_cost"),
        count(when(col("excessive_bench") == True, True)).alias("excessive_periods"),
        count(when(col("unexpected_bench") == True, True)).alias("unexpected_periods"),
        count(when(col("reason") == "Post-hire", True)).alias("post_hire_periods"),
        count(when(col("reason") == "Training", True)).alias("training_periods"),
        count(when(col("reason") == "Between projects", True)).alias("between_projects_periods"),
        count(when(col("reason") == "Internal", True)).alias("internal_periods")
    )
    .withColumn("bench_risk_flag", when(
        col("total_days_on_bench") > 180, lit("High risk"))
        .when(col("total_days_on_bench") > 90, lit("Medium risk"))
        .otherwise(lit("Low risk")))
    .withColumn("gold_loaded_at", current_timestamp())
)

# Practice area bench summary
practice_bench_df = (
    joined_df
    .groupBy("practice_area", "seniority")
    .agg(
        count("bench_id").alias("total_bench_periods"),
        round(avg("days_on_bench_clean"), 0).alias("avg_days_on_bench"),
        round(sum("estimated_bench_cost"), 2).alias("total_estimated_cost"),
        count(when(col("excessive_bench") == True, True)).alias("excessive_periods"),
        count(when(col("unexpected_bench") == True, True)).alias("unexpected_periods")
    )
    .withColumn("gold_loaded_at", current_timestamp())
)

# Write to Gold
employee_bench_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.bench_by_employee")
practice_bench_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.bench_by_practice")

print(f"Employee bench summary rows:  {employee_bench_df.count()}")
print(f"Practice bench summary rows:  {practice_bench_df.count()}")
print(f"Written to gold.bench_by_employee: {spark.table('gold.bench_by_employee').count()} rows")
print(f"Written to gold.bench_by_practice: {spark.table('gold.bench_by_practice').count()} rows")